# Lesson 13 - Agent Memory


## Setup

Dis notebook dey show how to build travel booking agent wey get **persistent memory** using **Microsoft Agent Framework** (MAF).

You go learn how different kain agent memory — working, short-term, and long-term — dey affect how agent dey keep and use information during conversations.

**Prerequisites:**
- Azure AI Foundry project wey get chat model wey dem don deploy (e.g. `gpt-4o-mini`).
- You don log in with Azure CLI — run `az login` for your terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — your Azure AI Foundry project endpoint.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Kain Kain Agent Memory

AI agents fit use different kain memory, each dey do different work:

### Working Memory
Na di conversation thread - di messages wey dem exchange for one session. Di agent fit look back di earlier messages for di same thread to keep am make sense. For MAF dis one dey created by **`agent.create_session()`**, wey go return one `AgentSession`.

### Short-Term Memory
Info wey go last for di time wey di task or session dey but no go stay permanent. For example, di agent fit gather facts during one multi-turn planning talk and use dem to produce di final itinerary.

### Long-Term Memory
Preferences and facts wey dey last **across sessions**. Person wey don come again no need to repeat dia dietary restrictions or travel style. Long-term memory dey usually backed by one external storage - database, file, or vector index - and di agent dey see am through tools.


## Working Memory wit Sessions

Di simplest kain memory na di conversation session. Wen you pass di same session (wey you create via `agent.create_session()`) to di next `agent.run()` calls, di agent go see di full history of di conversation and fit remember earlier details.

Make we create travel agent and show how working memory dey work.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

The agent correctly recall di budget because both messages get di same session. Dis na **working memory** — e dey only last for di time wey di session dey.

### Wetin happen wit new thread?

If we create **new** session, di agent no get memory of di previous conversation:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Long-Term Memory Pattern

To remember user preferences **across sessions**, we need a persistent store wey dey outside the conversation thread. The agent dey access this store through **tools** — na functions e fit call to save and collect information.

Below we go implement a simple in-memory preference store (for production, you go back am with database or vector index) and make e show as tools wey the agent fit use.

### Architecture
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — First-time user book anniversary trip

Sarah come visit first time. The agent suppose store her preferences through the tools and use dem to recommend hotels.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenario 2 — Sarah come back after some weeks

Sarah start one **brand-new thread** (wey dey simulate new session). Di working memory empty, but di long-term preference store still get her information. Di agent suppose find am and use am to personalize recommendations.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Summary

For dis lesson you learn three kain agent memory and how to implement dem wit di Microsoft Agent Framework:

| Memory Type | MAF Mechanism | Lifetime |
|---|---|---|
| **Working** | `agent.create_session()` | Single conversation |
| **Short-term** | Accumulated context within a thread | Single task / session |
| **Long-term** | External store accessed via `@tool` functions | Across sessions |

### Key Takeaways
1. **`agent.create_session()`** de provide working memory — di agent dey see di full conversation history inside one session.
2. **New sessions go lose context** — without long-term memory di agent no fit remember past talks.
3. **`@tool` functions** na bridge — dem dey allow di agent save and collect info from persistent store.
4. **Personalization dey improve over time** — di more preferences wey dem store, di better di agent recommendation go be.

### Real-World Applications
- **Customer Service**: Remember customer history and preferences
- **Personal Assistants**: Maintain context across days or weeks
- **Healthcare**: Track patient information and preferences
- **E-commerce**: Personalized shopping based on history

### Next Steps
- Replace di in-memory dict wit database or vector store (e.g. Azure AI Search)
- Add memory expiration for time-sensitive information
- Build multi-agent systems wit shared memory
- Explore di Cognee notebook for knowledge-graph-backed memory


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document na AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator) wey dem use translate am. Even though we dey try make am correct, abeg sabi say automated translation fit get some mistakes or no too correct. Di original document wey e dey im own language na di correct source. If na serious matter, e better make human translation person translate am well well. We no go responsible for any wahala or wrong understand wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
